In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhinavkum/india-national-family-health-survey-nfhs5")

print("Path to dataset files:", path)

In [ ]:
import os

dataset_path = r"C:\Users\Lenovo\.cache\kagglehub\datasets\abhinavkum\india-national-family-health-survey-nfhs5\versions\3"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_path}/National Family Health Survey (NFHS-5) 2019-20.csv")

df.head()

In [ ]:
df.columns = [
    "SNo",
    "IndicatorCode",
    "Indicator",
    "SubIndicator",
    "Urban",
    "Rural",
    "Total",
    "NFHS4_Total",
    "State"
]

In [ ]:
df = df[["State", "SubIndicator", "Total"]]
df.head()

In [ ]:
for col in health_df.columns[1:]:
    health_df[col] = (
        health_df[col]
        .astype(str)
        .str.replace(r"[()*]", "", regex=True)
        .replace(["na", "NA", "*"], pd.NA)
    )
    health_df[col] = pd.to_numeric(health_df[col], errors="coerce")

In [ ]:
health_df = selected.pivot(
    index="State",
    columns="SubIndicator",
    values="Total"
).reset_index()

health_df.head()

In [ ]:
health_df = pd.DataFrame()

health_df["State"] = sorted(df["State"].dropna().unique())

In [ ]:
def extract_indicator(keyword):
    temp = df[df["SubIndicator"].str.contains(keyword, case=False, na=False)]
    temp = temp[["State", "Total"]]
    return temp.set_index("State")["Total"]

In [ ]:
temp = df[df["SubIndicator"].str.contains(
    "Men who are overweight or obese",
    case=False,
    na=False
)]

print(temp.shape)
temp.head(20)

In [ ]:
temp["State"].value_counts().head(20)

In [ ]:
def extract_indicator(keyword):
    temp = df[df["SubIndicator"].str.contains(keyword, case=False, na=False)]

    temp = (
        temp[["State", "Total"]]
        .drop_duplicates(subset="State")
        .set_index("State")
    )

    return temp["Total"]

In [ ]:
print(df["State"].value_counts().head(20))

In [ ]:
temp = df[df["SubIndicator"].str.contains(
    "Men who are overweight or obese",
    case=False,
    na=False
)]

print(temp.shape)
print(temp["State"].value_counts().head())

In [ ]:
keywords = [
    "overweight or obese",
    "Children age 6-59 months who are anaemic",
    "All women age 15-49 years who are anaemic",
    "Blood sugar level - high or very high",
    "Elevated blood pressure"
]

filtered = df[
    df["SubIndicator"].str.contains("|".join(keywords), case=False, na=False)
]

print(filtered.shape)
filtered.head()

In [ ]:
health_df = filtered.pivot_table(
    index="State",
    columns="SubIndicator",
    values="Total",
    aggfunc="first"   # handles duplicates automatically
)

health_df = health_df.reset_index()

In [ ]:
print(health_df.columns.tolist())

In [ ]:
health_df = health_df.rename(columns={
    "All women age 15-49 years who are anaemic (%)": "Anaemia_Women",
    "Children age 6-59 months who are anaemic (<11.0 g/dl) (%)": "Anaemia_Children",
    "Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Men": "BloodSugar_Men",
    "Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Women": "BloodSugar_Women",
    "Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%) - Men": "Hypertension_Men",
    "Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%) - Women": "Hypertension_Women",
    "Men who are overweight or obese (BMI ≥25.0 kg/m2) (%)": "Obesity_Men",
    "Women who are overweight or obese (BMI ≥25.0 kg/m2) (%)": "Obesity_Women"
})

In [ ]:
health_df = health_df[
    [
        "State",
        "Obesity_Men",
        "Obesity_Women",
        "Anaemia_Children",
        "Anaemia_Women",
        "BloodSugar_Men",
        "BloodSugar_Women",
        "Hypertension_Men",
        "Hypertension_Women"
    ]
]

In [ ]:
import pandas as pd

for col in health_df.columns[1:]:
    health_df[col] = (
        health_df[col]
        .astype(str)
        .str.replace(r"[()*]", "", regex=True)
        .replace(["na", "NA", "*"], pd.NA)
    )
    health_df[col] = pd.to_numeric(health_df[col], errors="coerce")

In [ ]:
health_df["Gender_Obesity_Gap"] = (
    health_df["Obesity_Women"] - health_df["Obesity_Men"]
)

In [ ]:
health_df.head()

In [ ]:
health_df.info()

In [ ]:
health_df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top10 = health_df.sort_values(
    by="Hypertension_Women",
    ascending=False
).head(10)

x = np.arange(len(top10))
width = 0.35

plt.figure(figsize=(12,6))

plt.bar(
    x - width/2,
    top10["Hypertension_Men"],
    width,
    label="Men"
)

plt.bar(
    x + width/2,
    top10["Hypertension_Women"],
    width,
    label="Women"
)

plt.xticks(
    x,
    top10["State"],
    rotation=45,
    ha="right"
)

plt.ylabel("Hypertension (%)")
plt.xlabel("State")
plt.title("Top 10 States with Highest Hypertension Rates")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TASK 1 : Composite Health Risk Index
# ============================================================

import numpy as np
import pandas as pd

risk_df = health_df.copy()

risk_columns = [
    "Obesity_Women",
    "Obesity_Men",
    "Anaemia_Children",
    "Anaemia_Women",
    "BloodSugar_Women",
    "BloodSugar_Men",
    "Hypertension_Women",
    "Hypertension_Men"
]
print("Missing Values")
print(risk_df[risk_columns].isnull().sum())

# -----------------------------
# Min-Max Normalization
# -----------------------------

for col in risk_columns:
    risk_df[col] = (
        risk_df[col] - risk_df[col].min()
    ) / (
        risk_df[col].max() - risk_df[col].min()
    )

# -----------------------------
# Composite Health Risk Score
# -----------------------------

risk_df["Composite_Risk"] = (
      0.20*risk_df["Obesity_Women"]
    + 0.15*risk_df["Obesity_Men"]
    + 0.20*risk_df["Anaemia_Children"]
    + 0.15*risk_df["Anaemia_Women"]
    + 0.10*risk_df["BloodSugar_Women"]
    + 0.10*risk_df["BloodSugar_Men"]
    + 0.05*risk_df["Hypertension_Women"]
    + 0.05*risk_df["Hypertension_Men"]
)

# -----------------------------
# Risk Categories
# -----------------------------

q1 = risk_df["Composite_Risk"].quantile(.25)
q2 = risk_df["Composite_Risk"].quantile(.50)
q3 = risk_df["Composite_Risk"].quantile(.75)

def assign(x):
    if x>=q3:
        return "Critical"
    elif x>=q2:
        return "High"
    elif x>=q1:
        return "Moderate"
    return "Low"

risk_df["Risk_Tier"]=risk_df["Composite_Risk"].apply(assign)

# -----------------------------
# Ranking
# -----------------------------

risk_df = risk_df.sort_values(
    "Composite_Risk",
    ascending=False
).reset_index(drop=True)

risk_df["Rank"] = risk_df.index + 1
risk_df["Anaemia_NCD_Ratio"] = (
    risk_df["Anaemia_Women"] /
    (
        risk_df["BloodSugar_Women"]
        + risk_df["BloodSugar_Men"]
        + risk_df["Hypertension_Women"]
        + risk_df["Hypertension_Men"]
        + 1e-9
    )
)

print("\nTop 5 Critical States")
print(
    risk_df[risk_df["Risk_Tier"]=="Critical"]
    .nlargest(5,"Composite_Risk")
    [["State","Composite_Risk"]]
)

# -----------------------------
# Final Output
# -----------------------------

final_result = risk_df[
    [
        "Rank",
        "State",
        "Composite_Risk",
        "Risk_Tier"
    ]
]

display(final_result)

# Save CSV
final_result.to_csv(
    "Composite_Health_Risk_Index.csv",
    index=False
)

print("CSV saved successfully.")

In [ ]:
# ============================================================
# TASK 2 : State Clustering using Graph + BFS
# ============================================================

from collections import deque

# ---------------------------------------------
# Use dataframe from Task 1
# ---------------------------------------------

cluster_df = risk_df.copy()

# Similarity threshold
THRESHOLD = 0.05

# ---------------------------------------------
# Build Graph (Adjacency List)
# ---------------------------------------------

graph = {}

states = cluster_df["State"].tolist()
scores = cluster_df["Composite_Risk"].tolist()

for state in states:
    graph[state] = []

for i in range(len(states)):
    for j in range(i + 1, len(states)):

        if abs(scores[i] - scores[j]) <= THRESHOLD:

            graph[states[i]].append(states[j])
            graph[states[j]].append(states[i])

# ---------------------------------------------
# Breadth First Search
# ---------------------------------------------

visited = set()
clusters = []
cluster_map = {}

for state in graph:

    if state not in visited:

        queue = deque([state])
        visited.add(state)

        current_cluster = []

        while queue:

            node = queue.popleft()
            current_cluster.append(node)

            for neighbour in graph[node]:

                if neighbour not in visited:
                    visited.add(neighbour)
                    queue.append(neighbour)

        clusters.append(current_cluster)

        cluster_id = len(clusters)

        for s in current_cluster:
            cluster_map[s] = cluster_id

# ---------------------------------------------
# Store Cluster IDs
# ---------------------------------------------

cluster_df["BFS_Cluster"] = cluster_df["State"].map(cluster_map)

# Make available for later tasks
risk_df["BFS_Cluster"] = risk_df["State"].map(cluster_map)

# ---------------------------------------------
# Cluster Statistics
# ---------------------------------------------

largest_cluster = max(clusters, key=len)

isolated_states = [
    state
    for state, neighbours in graph.items()
    if len(neighbours) == 0
]

# Validation

assert len(cluster_map) == len(states), "Some states were not assigned a cluster."
assert cluster_df["BFS_Cluster"].isna().sum() == 0, "Missing BFS Cluster IDs."

# ---------------------------------------------
# Display Results
# ---------------------------------------------

print("=" * 60)
print("STATE HEALTH CLUSTERS")
print("=" * 60)

for i, cluster in enumerate(clusters, start=1):

    print(f"\nCluster {i}")

    for state in sorted(cluster):
        print("  •", state)

# ---------------------------------------------
# Summary
# ---------------------------------------------

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Total States         : {len(states)}")
print(f"Total Clusters Found : {len(clusters)}")

for i, cluster in enumerate(clusters, start=1):
    print(f"Cluster {i}: {len(cluster)} states")

print("\nLargest Cluster")
print(largest_cluster)

print("\nIsolated States")
print(isolated_states)

print("\nBFS Cluster IDs")
print(cluster_df[["State", "BFS_Cluster"]].sort_values("BFS_Cluster"))

In [ ]:
# ============================================================
# TASK 3 : Health Atlas using Matplotlib
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------
# Create 2x2 Figure
# ----------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# ==========================================================
# Plot 1 : Top 10 Highest Risk States
# ==========================================================

top10 = risk_df.nlargest(10, "Composite_Risk")

axes[0, 0].barh(
    top10["State"],
    top10["Composite_Risk"]
)

axes[0, 0].invert_yaxis()
axes[0, 0].set_title("Top 10 Highest Risk States")
axes[0, 0].set_xlabel("Composite Risk Index")
axes[0, 0].set_ylabel("State")

# ==========================================================
# Plot 2 : Scatter Plot
# Anaemia in Children vs Female Obesity
# ==========================================================

scatter = axes[0, 1].scatter(
    risk_df["Anaemia_Children"],
    risk_df["Obesity_Women"],
    s=risk_df["Composite_Risk"] * 200,
    c=risk_df["BFS_Cluster"],
    alpha=0.75
)

for _, row in risk_df.iterrows():

    axes[0, 1].text(
        row["Anaemia_Children"],
        row["Obesity_Women"],
        row["State"][:2].upper(),
        fontsize=6
    )

axes[0, 1].set_title("Anaemia (Children) vs Female Obesity")
axes[0, 1].set_xlabel("Child Anaemia (%)")
axes[0, 1].set_ylabel("Female Obesity (%)")

plt.colorbar(
    scatter,
    ax=axes[0, 1],
    label="BFS Cluster"
)

# ==========================================================
# Plot 3 : Correlation Heatmap
# ==========================================================

risk_columns = [
    "Obesity_Women",
    "Obesity_Men",
    "Anaemia_Children",
    "Anaemia_Women",
    "BloodSugar_Women",
    "BloodSugar_Men",
    "Hypertension_Women",
    "Hypertension_Men"
]

corr = risk_df[risk_columns].corr()

heatmap = axes[1, 0].imshow(corr)

axes[1, 0].set_xticks(range(len(corr.columns)))
axes[1, 0].set_xticklabels(
    corr.columns,
    rotation=90,
    fontsize=8
)

axes[1, 0].set_yticks(range(len(corr.columns)))
axes[1, 0].set_yticklabels(
    corr.columns,
    fontsize=8
)

for i in range(len(corr)):
    for j in range(len(corr)):
        axes[1, 0].text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )

axes[1, 0].set_title("Correlation Heatmap")

plt.colorbar(
    heatmap,
    ax=axes[1, 0]
)

# ==========================================================
# Plot 4 : Radar Chart
# ==========================================================

fig.delaxes(axes[1, 1])

ax = fig.add_subplot(224, polar=True)

radar_cols = [
    "Obesity_Women",
    "Obesity_Men",
    "Anaemia_Children",
    "Anaemia_Women"
]

angles = np.linspace(
    0,
    2 * np.pi,
    len(radar_cols),
    endpoint=False
)

angles = np.concatenate((angles, [angles[0]]))

for tier in risk_df["Risk_Tier"].unique():

    values = (
        risk_df[risk_df["Risk_Tier"] == tier][radar_cols]
        .mean()
        .tolist()
    )

    values.append(values[0])

    ax.plot(
        angles,
        values,
        linewidth=2,
        label=tier
    )

    ax.fill(
        angles,
        values,
        alpha=0.1
    )

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_cols, fontsize=8)

ax.set_title("Average Health Indicators by Risk Tier")
ax.legend(loc="upper right")

# ==========================================================
# Overall Figure
# ==========================================================

plt.suptitle(
    "NFHS-5 India Health Atlas",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "health_atlas.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
with open("india_state.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

In [ ]:
# ============================================================
# TASK 4 : Interactive Health Risk Dashboard using Folium
# ============================================================

import folium

# ----------------------------------------------------------
# Prepare Data
# ----------------------------------------------------------

map_df = risk_df.copy()

state_mapping = {
    "Andaman & Nicobar Islands": "Andaman and Nicobar",
    "Dadra & Nagar Haveli and Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",
    "NCT of Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Odisha": "Orissa"
}

map_df["State"] = map_df["State"].replace(state_mapping)

map_df = map_df[
    ~map_df["State"].isin([
        "Lakshadweep",
        "Ladakh"
    ])
]

# ----------------------------------------------------------
# Create Base Map
# ----------------------------------------------------------

india_map = folium.Map(
    location=[22.5, 79],
    zoom_start=5,
    tiles="CartoDB positron"
)

# ----------------------------------------------------------
# Choropleth Layer
# ----------------------------------------------------------

folium.Choropleth(
    geo_data="india_state.geojson",
    data=map_df,
    columns=["State", "Composite_Risk"],
    key_on="feature.properties.NAME_1",
    fill_color="YlOrRd",
    fill_opacity=0.8,
    line_opacity=0.4,
    legend_name="Composite Health Risk Index"
).add_to(india_map)

# ----------------------------------------------------------
# Risk Indicator Columns
# ----------------------------------------------------------

risk_columns = [
    "Obesity_Women",
    "Obesity_Men",
    "Anaemia_Children",
    "Anaemia_Women",
    "BloodSugar_Women",
    "BloodSugar_Men",
    "Hypertension_Women",
    "Hypertension_Men"
]

# ----------------------------------------------------------
# Circle Markers
# ----------------------------------------------------------

for _, row in map_df.iterrows():

    state = row["State"]

    lat = None
    lon = None

    for feature in geojson_data["features"]:

        if feature["properties"]["NAME_1"] == state:

            coords = feature["geometry"]["coordinates"]

            if feature["geometry"]["type"] == "Polygon":

                lon = sum(p[0] for p in coords[0]) / len(coords[0])
                lat = sum(p[1] for p in coords[0]) / len(coords[0])

            else:

                largest = max(coords, key=len)

                lon = sum(p[0] for p in largest[0]) / len(largest[0])
                lat = sum(p[1] for p in largest[0]) / len(largest[0])

            break

    if lat is None:
        continue

    # ------------------------------------------------------
    # Marker Color
    # ------------------------------------------------------

    if row["Risk_Tier"] == "Critical":
        color = "darkred"

    elif row["Risk_Tier"] == "High":
        color = "red"

    elif row["Risk_Tier"] == "Moderate":
        color = "orange"

    else:
        color = "green"

    # ------------------------------------------------------
    # Top Two Indicators
    # ------------------------------------------------------

    top2 = (
        row[risk_columns]
        .sort_values(ascending=False)
        .head(2)
        .index
        .tolist()
    )

    top2_text = ", ".join(top2)

    # ------------------------------------------------------
    # Popup
    # ------------------------------------------------------

    popup = f"""
    <b>State:</b> {row['State']}<br>
    <b>Composite Risk:</b> {row['Composite_Risk']:.3f}<br>
    <b>Risk Tier:</b> {row['Risk_Tier']}<br>
    <b>BFS Cluster:</b> {row['BFS_Cluster']}<br>
    <b>Top Indicators:</b> {top2_text}
    """

    # ------------------------------------------------------
    # Marker
    # ------------------------------------------------------

    folium.CircleMarker(
        location=[lat, lon],
        radius=max(5, row["Composite_Risk"] * 30),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=300)
    ).add_to(india_map)

# ----------------------------------------------------------
# Layer Control
# ----------------------------------------------------------

folium.LayerControl().add_to(india_map)

# ----------------------------------------------------------
# Save Dashboard
# ----------------------------------------------------------

india_map.save("health_risk_map.html")

print("Dashboard saved as health_risk_map.html")

india_map